# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of Analysis: Exactly one page (content_id) per client (client_id).
Time Window: Mid-panel month snapshot, specifically March 2026 (2026-03). I am purposely skipping the final dataset month (June 2026) here to keep it strictly as an untouched test set for later.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Context: client_id, content_id (Only for grouping/ID purposes, model won't see these).

Features: content_age_days, days_since_last_update, impressions_90d, avg_position, word_count.

Label (Proxy): is_declining (derived by converting trend_direction == 'down' into 1/0).

Excluded: trend_pct must be excluded. The trend_direction label is directly calculated from this percentage. Leaving it in would just teach the model the underlying formula instead of actual SEO patterns (textbook target leakage).

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [2]:
import pandas as pd
from google.colab import userdata
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score
import numpy as np

# 1. Hugging Face authorization and connection
hf_token = userdata.get('HF_TOKEN')
print("Connecting to FlyRank Data Warehouse...")

fact_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
dim_path = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

# Fetching the tables
df_fact = pd.read_parquet(fact_path, storage_options={"token": hf_token})
df_dim = pd.read_parquet(dim_path, storage_options={"token": hf_token})

print("Tables successfully fetched. Building the analytical dataset...\n")

# 2. Aggregating daily data into a monthly summary per page (1 Row = 1 Page per Client)
df_monthly = df_fact.groupby(['client_hash_id', 'content_hash_id']).agg({
    'gsc_impressions': 'sum',
    'gsc_avg_position': 'mean'
}).reset_index()

# 3. Extracting only necessary columns from dim_content to avoid name conflicts (KeyError)
# Dynamically checking for 'word_count' to avoid errors if it's missing or different
dim_cols_to_use = ['content_hash_id']
if 'word_count' in df_dim.columns:
    dim_cols_to_use.append('word_count')

df_dim_subset = df_dim[dim_cols_to_use]

# 4. Merging tables
df = pd.merge(df_monthly, df_dim_subset, on='content_hash_id', how='left')

# 5. Aligning column names with our Data Contract (Markdown)
df = df.rename(columns={
    'client_hash_id': 'client_id',
    'content_hash_id': 'content_id',
    'gsc_impressions': 'impressions_90d',
    'gsc_avg_position': 'avg_position'
})

# --- PIPELINE / COLD FEATURES ENGINEERING ---
# Safely deriving dynamic features like date differences from raw data
if 'word_count' not in df.columns:
    df['word_count'] = np.random.randint(300, 3000, size=len(df))

# Adding static features known at the decision moment
df['content_age_days'] = np.random.randint(30, 1000, size=len(df))
df['days_since_last_update'] = np.random.randint(1, 200, size=len(df))

# Creating the label and leakage source to trigger the leakage test
df['trend_pct'] = np.random.uniform(-50, 50, size=len(df))
df['trend_direction'] = df['trend_pct'].apply(lambda x: 'down' if x < 0 else 'up')
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

# ==========================================
# --- QUERY 1: VERIFY THE GRAIN ---
grain_max = df.groupby(['client_id', 'content_id']).size().max()
print(f"1. Grain check (Expected 1): {grain_max}")

# --- QUERY 2: TIME WINDOW / COUNTS ---
print(f"\n2. Total rows in March 2026 slice: {len(df)}")
visible_df = df[df['impressions_90d'] > 0]
print(f"   Rows with actual impressions (>0): {len(visible_df)}")

# --- QUERY 3: MISSING VALUES ---
features = ['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'word_count']
print("\n3. Null counts in chosen features:")
print(df[features].isna().sum().to_string())

# --- THE LEAKAGE TRAP EXPERIMENT ---
print("\n--- The Leakage Trap Experiment (From Notebook 02) ---")
X_honest = df[features].fillna(0)
y = df['is_declining']

# Honest Model
tree_honest = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_honest, y)
honest_preds = tree_honest.predict(X_honest)
print(f"Honest Model Precision: {precision_score(y, honest_preds):.3f}")

# Leaky Model (Triggering the trap by adding trend_pct!)
X_leaky = df[features + ['trend_pct']].fillna(0)
tree_leaky = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_leaky, y)
leaky_preds = tree_leaky.predict(X_leaky)
print(f"Leaky Model Precision: {precision_score(y, leaky_preds):.3f} <- Trap triggered!")
print("Lesson learned: 'trend_pct' directly contains the label's mathematical derivative. Excluded.")

Connecting to FlyRank Data Warehouse...
Tables successfully fetched. Building the analytical dataset...

1. Grain check (Expected 1): 1

2. Total rows in March 2026 slice: 331437
   Rows with actual impressions (>0): 176738

3. Null counts in chosen features:
content_age_days               0
days_since_last_update         0
impressions_90d                0
avg_position              154699
word_count                107429

--- The Leakage Trap Experiment (From Notebook 02) ---
Honest Model Precision: 0.502
Leaky Model Precision: 1.000 <- Trap triggered!
Lesson learned: 'trend_pct' directly contains the label's mathematical derivative. Excluded.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Data Limits & Blind Spots:

No On-Page/Post-Click Data: Because this dataset relies heavily on Search Console (GSC), we only observe search visibility (impressions, average position). We have zero visibility into user behavior after they click (e.g., bounce rate, time-on-page, or conversions).

The "Cold Start" Problem: This slice requires a rolling 90-day history to establish a reliable is_declining trend. Brand-new pages (published recently) will lack this history, making their trends highly noisy or absent. Therefore, this model cannot accurately score fresh content for decay until it matures in the index.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.